# 1. Data processing

This notebook assembles the two analysis-ready panels used by resource demand and equilibrium analysis:

1. a daily execution/data/state panel from 2025-01-01 through 2026-05-31; and
2. a 120-day February–May 2026 accounting panel and a 50-block sample per day for the metering and equilibrium anchors.

The underlying measurements come from Xatu's raw and CBT ClickHouse services plus a deterministic Erigon-compatible RPC sample. Large query caches and RPC-derived block samples live under `data/` and are intentionally ignored by Git.
Set `REFRESH_FROM_NETWORK = True` below to query Xatu/CBT, execute the 6,000-block RPC calibration, reconstruct the full refund panel, and recreate every required export before processing.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.6f}".format


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = DATA_DIR / "glamsterdam"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DAILY_OUT = OUT_DIR / "daily_resource_panel_2025-01-01_2026-05-31.csv"
ANCHOR_OUT = OUT_DIR / "anchor_accounting_panel_2026-02-01_2026-05-31.csv"
MANIFEST_OUT = OUT_DIR / "source_manifest.csv"

PROJECT_ROOT

PosixPath('/Users/william/PycharmProjects/eip-7999-research')

## Full refresh from Xatu and RPC

The default is cache mode so routine reproduction does not repeat expensive network work. To rebuild every source export, set `REFRESH_FROM_NETWORK = True` and provide a local `.env` file with:

- `CLICKHOUSE_USER` and `CLICKHOUSE_PASSWORD`;
- optionally `CLICKHOUSE_RAW_HOST`, `CLICKHOUSE_CBT_HOST`, and `CLICKHOUSE_PORT`;
- either `ETHNODEOPS_API_KEY` (Erigon) or a complete `ALCHEMY_RPC` URL.

A full refresh queries Xatu/CBT for daily block, transaction, state, trace, and refund aggregates, then samples 50 blocks per day from a deterministic 500-block-per-day plan for RPC-only fields. The 120-day RPC run therefore contains 6,000 blocks. Network pulls are cached incrementally under `data/`.

The refresh is explicit because it is large and may take hours. There is no automatic retry of failed RPC blocks.

In [2]:
import os
import time
import sys

import clickhouse_connect
from dotenv import load_dotenv

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.rpc_calibration import calibrate_blocks, sample_blocks_per_day
from sim.xatu_refunds import query_xatu_refund_daily_full

REFRESH_FROM_NETWORK = False
NETWORK = "mainnet"
PECTRA_TS = "2025-05-07 10:05:00"
EVENT_START = "2025-01-01"
BRIDGE_START = "2026-01-05"
ANCHOR_START = "2026-02-01"
END_DATE = "2026-06-01"
CPSB = 1530

RPC_PLAN_PER_DAY = 500
RPC_BLOCKS_PER_DAY = 50
RPC_SEED = 42
RPC_MAX_WORKERS = 32
RPC_CHUNK_SIZE = 100

RAW_CACHE_DIR = OUT_DIR / "raw"
RAW_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def clickhouse_clients():
    load_dotenv(PROJECT_ROOT / ".env")
    required = ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"]
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        raise RuntimeError(
            "Full refresh requires these .env values: " + ", ".join(missing)
        )
    common = {
        "port": int(os.environ.get("CLICKHOUSE_PORT", "443")),
        "secure": True,
        "username": os.environ["CLICKHOUSE_USER"],
        "password": os.environ["CLICKHOUSE_PASSWORD"],
    }
    raw = clickhouse_connect.get_client(
        host=os.environ.get(
            "CLICKHOUSE_RAW_HOST",
            "clickhouse-raw.xatu.ethpandaops.io",
        ),
        **common,
    )
    cbt = clickhouse_connect.get_client(
        host=os.environ.get(
            "CLICKHOUSE_CBT_HOST",
            "clickhouse-cbt.xatu.ethpandaops.io",
        ),
        **common,
    )
    return raw, cbt


def rpc_configuration():
    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.environ.get("ETHNODEOPS_API_KEY") or os.environ.get("hoodi_api_key")
    alchemy = os.environ.get("ALCHEMY_RPC")
    if api_key:
        return (
            os.environ.get(
                "ETHNODEOPS_RPC",
                "https://erigon.mainnet.rpc.ethnodeops.xyz",
            ),
            {"X-API-Key": api_key},
            "ethnodeops_erigon_mainnet",
        )
    if alchemy:
        return alchemy, None, "alchemy_mainnet"
    raise RuntimeError(
        "Full refresh requires ETHNODEOPS_API_KEY, hoodi_api_key, or ALCHEMY_RPC"
    )


def normalize_dates(frame):
    out = frame.copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
    return out


def cached_query(path, query_fn, *, refresh):
    if path.exists() and not refresh:
        print("loaded", path.relative_to(PROJECT_ROOT))
        return normalize_dates(pd.read_csv(path))
    out = normalize_dates(query_fn())
    out.to_csv(path, index=False)
    print("wrote", path.relative_to(PROJECT_ROOT))
    return out

print("mode:", "full network refresh" if REFRESH_FROM_NETWORK else "cached exports")

mode: cached exports


## Xatu and CBT daily extraction

The raw Xatu query joins canonical block headers to canonical transaction receipts and computes current-rule calldata gas, including the EIP-7623 floor after Pectra. CBT supplies daily state-inventory boundaries. The event-study state proxy uses inventory-byte changes; the 2026 extension uses account-count, storage-byte, and contract-code-byte changes, matching the report's original accounting splice.

In [3]:
def query_block_bounds(raw_client, start_date, end_date):
    frame = raw_client.query_df(
        """
        SELECT
            min(block_number) AS min_block,
            max(block_number) AS max_block,
            count() AS block_count
        FROM default.canonical_execution_block FINAL
        WHERE meta_network_name = {network:String}
          AND block_date_time >= parseDateTime64BestEffort({start_ts:String})
          AND block_date_time < parseDateTime64BestEffort({end_ts:String})
        """,
        parameters={
            "network": NETWORK,
            "start_ts": f"{start_date} 00:00:00",
            "end_ts": f"{end_date} 00:00:00",
        },
    )
    return {
        "min_block": int(frame.loc[0, "min_block"]),
        "max_block": int(frame.loc[0, "max_block"]),
        "block_count": int(frame.loc[0, "block_count"]),
    }


def query_daily_execution(raw_client, start_date, end_date):
    bounds = query_block_bounds(raw_client, start_date, end_date)
    params = {
        "network": NETWORK,
        "start_ts": f"{start_date} 00:00:00",
        "end_ts": f"{end_date} 00:00:00",
        "pectra_ts": PECTRA_TS,
        "min_block": bounds["min_block"],
        "max_block": bounds["max_block"],
    }
    frame = raw_client.query_df(
        """
        WITH
            blocks AS
            (
                SELECT
                    block_number,
                    block_date_time,
                    toDate(block_date_time) AS date,
                    gas_used AS block_gas_used,
                    gas_limit AS block_gas_limit,
                    base_fee_per_gas
                FROM default.canonical_execution_block FINAL
                WHERE meta_network_name = {network:String}
                  AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
                  AND block_date_time >= parseDateTime64BestEffort({start_ts:String})
                  AND block_date_time < parseDateTime64BestEffort({end_ts:String})
            ),
            tx AS
            (
                SELECT
                    block_number,
                    count() AS tx_count,
                    sum(gas_used) AS receipt_gas_used,
                    sum(greatest(toInt64(gas_used) - 21000, 0)) AS receipt_body_gas,
                    sum(toUInt64(n_input_bytes)) AS calldata_bytes,
                    sum(toUInt64(n_input_zero_bytes)) AS calldata_zero_bytes,
                    sum(toUInt64(n_input_nonzero_bytes)) AS calldata_nonzero_bytes,
                    sum(toInt64(4 * (n_input_zero_bytes + 4 * n_input_nonzero_bytes)))
                        AS standard_calldata_gas,
                    sum(
                        if(
                            greatest(toInt64(gas_used) - 21000, 0)
                                <= toInt64(10 * (n_input_zero_bytes + 4 * n_input_nonzero_bytes))
                            AND (n_input_zero_bytes + n_input_nonzero_bytes) > 0,
                            toInt64(10 * (n_input_zero_bytes + 4 * n_input_nonzero_bytes)),
                            toInt64(4 * (n_input_zero_bytes + 4 * n_input_nonzero_bytes))
                        )
                    ) AS post_7623_data_gas,
                    countIf(
                        greatest(toInt64(gas_used) - 21000, 0)
                            <= toInt64(10 * (n_input_zero_bytes + 4 * n_input_nonzero_bytes))
                        AND (n_input_zero_bytes + n_input_nonzero_bytes) > 0
                    ) AS floor_bound_7623_proxy_txs
                FROM default.canonical_execution_transaction FINAL
                WHERE meta_network_name = {network:String}
                  AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
                GROUP BY block_number
            ),
            by_block AS
            (
                SELECT
                    b.*,
                    ifNull(t.tx_count, 0) AS tx_count,
                    ifNull(t.receipt_gas_used, 0) AS receipt_gas_used,
                    ifNull(t.receipt_body_gas, 0) AS receipt_body_gas,
                    ifNull(t.calldata_bytes, 0) AS calldata_bytes,
                    ifNull(t.calldata_zero_bytes, 0) AS calldata_zero_bytes,
                    ifNull(t.calldata_nonzero_bytes, 0) AS calldata_nonzero_bytes,
                    ifNull(t.standard_calldata_gas, 0) AS standard_calldata_gas,
                    if(
                        b.block_date_time >= parseDateTime64BestEffort({pectra_ts:String}),
                        ifNull(t.post_7623_data_gas, 0),
                        ifNull(t.standard_calldata_gas, 0)
                    ) AS data_gas_current,
                    if(
                        b.block_date_time >= parseDateTime64BestEffort({pectra_ts:String}),
                        ifNull(t.post_7623_data_gas, 0)
                            - ifNull(t.standard_calldata_gas, 0),
                        0
                    ) AS eip7623_data_uplift_proxy,
                    if(
                        b.block_date_time >= parseDateTime64BestEffort({pectra_ts:String}),
                        ifNull(t.floor_bound_7623_proxy_txs, 0),
                        0
                    ) AS floor_bound_7623_proxy_txs
                FROM blocks AS b
                LEFT JOIN tx AS t USING block_number
            )
        SELECT
            date,
            count() AS block_count,
            min(block_number) AS min_block,
            max(block_number) AS max_block,
            sum(block_gas_used) AS block_gas_used,
            sum(receipt_gas_used) AS receipt_gas_used,
            sum(receipt_body_gas) AS receipt_body_gas,
            sum(tx_count) AS tx_count,
            avg(toFloat64(block_gas_limit)) AS mean_gas_limit,
            quantileExact(0.5)(block_gas_limit) AS median_gas_limit,
            avg(toFloat64(base_fee_per_gas)) AS mean_base_fee_per_gas,
            quantileExact(0.5)(toFloat64(base_fee_per_gas))
                AS median_base_fee_per_gas,
            sum(toFloat64(base_fee_per_gas) * toFloat64(block_gas_used))
                / nullIf(sum(toFloat64(block_gas_used)), 0)
                AS gas_weighted_base_fee_per_gas,
            sum(calldata_bytes) AS calldata_bytes,
            sum(calldata_zero_bytes) AS calldata_zero_bytes,
            sum(calldata_nonzero_bytes) AS calldata_nonzero_bytes,
            sum(standard_calldata_gas) AS standard_calldata_gas,
            sum(data_gas_current) AS data_gas_current,
            sum(eip7623_data_uplift_proxy) AS eip7623_data_uplift_proxy,
            sum(floor_bound_7623_proxy_txs) AS floor_bound_7623_proxy_txs
        FROM by_block
        GROUP BY date
        ORDER BY date
        """,
        parameters=params,
        settings={"max_execution_time": 1200},
    )
    return frame, bounds


def query_daily_state_inventory(cbt_client, start_date, end_date, bounds):
    return cbt_client.query_df(
        """
        SELECT
            toDate(b.block_date_time) AS date,
            argMin(s.accounts, s.block_number) AS start_accounts,
            argMax(s.accounts, s.block_number) AS end_accounts,
            argMin(s.storages, s.block_number) AS start_storages,
            argMax(s.storages, s.block_number) AS end_storages,
            argMin(s.account_bytes, s.block_number) AS start_account_bytes,
            argMax(s.account_bytes, s.block_number) AS end_account_bytes,
            argMin(s.contract_code_bytes, s.block_number)
                AS start_contract_code_bytes,
            argMax(s.contract_code_bytes, s.block_number)
                AS end_contract_code_bytes,
            argMin(s.storage_bytes, s.block_number) AS start_storage_bytes,
            argMax(s.storage_bytes, s.block_number) AS end_storage_bytes
        FROM mainnet.int_execution_state_size_by_block AS s
        GLOBAL INNER JOIN
        (
            SELECT block_number, block_date_time
            FROM mainnet.int_execution_block_by_date
            WHERE block_date_time >= parseDateTime64BestEffort({start_ts:String})
              AND block_date_time < parseDateTime64BestEffort({end_ts:String})
              AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
        ) AS b USING block_number
        WHERE s.block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
        GROUP BY date
        ORDER BY date
        """,
        parameters={
            "start_ts": f"{start_date} 00:00:00",
            "end_ts": f"{end_date} 00:00:00",
            "min_block": bounds["min_block"],
            "max_block": bounds["max_block"],
        },
        settings={"max_execution_time": 600},
    )


def assigned_limit_mgas(date):
    date = pd.Timestamp(date)
    if date < pd.Timestamp("2025-02-04"):
        return 30.0
    if date < pd.Timestamp("2025-07-21"):
        return 36.0
    if date < pd.Timestamp("2025-11-25"):
        return 45.0
    return 60.0


def refresh_daily_exports(raw_client, cbt_client):
    execution, bounds = query_daily_execution(
        raw_client,
        EVENT_START,
        END_DATE,
    )
    inventory = query_daily_state_inventory(
        cbt_client,
        EVENT_START,
        END_DATE,
        bounds,
    )
    execution = normalize_dates(execution)
    inventory = normalize_dates(inventory)
    daily = execution.merge(inventory, on="date", how="inner", validate="one_to_one")

    for field in [
        "accounts",
        "storages",
        "account_bytes",
        "contract_code_bytes",
        "storage_bytes",
    ]:
        daily[f"{field}_net_delta"] = (
            daily[f"end_{field}"] - daily[f"start_{field}"]
        )

    event = daily[daily["date"] < pd.Timestamp(BRIDGE_START)].copy()
    event["gas_limit_mgas"] = event["date"].map(assigned_limit_mgas)
    event["current_state_inventory_delta_gas_proxy"] = (
        event["account_bytes_net_delta"].clip(lower=0) * (25_000 / 112)
        + event["storage_bytes_net_delta"].clip(lower=0) * (20_000 / 32)
        + event["contract_code_bytes_net_delta"].clip(lower=0) * 200
    )
    event_path = DATA_DIR / (
        "gas_limit_data_share_daily_2025-01-01_2026-01-05.csv"
    )
    event.to_csv(event_path, index=False)

    current_columns = [
        "date",
        "block_count",
        "tx_count",
        "receipt_gas_used",
        "receipt_body_gas",
        "calldata_bytes",
        "calldata_zero_bytes",
        "calldata_nonzero_bytes",
        "standard_calldata_gas",
        "data_gas_current",
        "eip7623_data_uplift_proxy",
        "floor_bound_7623_proxy_txs",
    ]
    for start_date, end_date in [
        (BRIDGE_START, ANCHOR_START),
        (ANCHOR_START, END_DATE),
    ]:
        frame = daily[
            (daily["date"] >= pd.Timestamp(start_date))
            & (daily["date"] < pd.Timestamp(end_date))
        ]
        path = DATA_DIR / f"daily_current_data_gas_xatu_{start_date}_{end_date}.csv"
        frame[current_columns].to_csv(path, index=False)

    bridge = daily[
        (daily["date"] >= pd.Timestamp(BRIDGE_START))
        & (daily["date"] < pd.Timestamp(ANCHOR_START))
    ].copy()
    bridge["current_gas_used"] = bridge["block_gas_used"]
    bridge_path = DATA_DIR / (
        f"daily_accounting_panel_{BRIDGE_START}_{ANCHOR_START}.csv"
    )
    bridge.to_csv(bridge_path, index=False)

    raw_daily_path = RAW_CACHE_DIR / (
        "xatu_cbt_daily_2025-01-01_2026-06-01.csv"
    )
    daily.to_csv(raw_daily_path, index=False)
    return daily


if REFRESH_FROM_NETWORK:
    raw_client, cbt_client = clickhouse_clients()
    print("raw Xatu:", raw_client.query("SELECT version()").result_rows)
    print("CBT:", cbt_client.query("SELECT version()").result_rows)
    refreshed_daily = refresh_daily_exports(raw_client, cbt_client)
    print("refreshed daily rows:", len(refreshed_daily))
else:
    raw_client = None
    cbt_client = None
    print("Daily Xatu/CBT refresh skipped; existing exports will be used.")

Daily Xatu/CBT refresh skipped; existing exports will be used.


## Anchor-specific Xatu pulls and deterministic RPC calibration

The February–May anchor additionally requires:

- Xatu storage-slot and contract-code creation counts;
- the transaction-level EIP-7976 floor uplift; and
- RPC-only access lists, authorization tuples, exact new-account counts, and delegation indicators.

The sample plan draws 500 candidate blocks per day with seed 42. The executed subset is the first 50 stable ranks per day, so reruns and later top-ups never resample earlier blocks.

In [4]:
def query_eip7976_daily(raw_client, bounds):
    return raw_client.query_df(
        """
        WITH tx_by_block AS
        (
            SELECT
                block_number,
                sum(
                    greatest(
                        0,
                        (64 * n_input_bytes)
                            - greatest(toInt64(gas_used) - 21000, 0)
                    )
                ) AS eip7976_floor_uplift_current_body,
                countIf(
                    (64 * n_input_bytes)
                        > greatest(toInt64(gas_used) - 21000, 0)
                ) AS eip7976_floor_bound_txs_current_body
            FROM default.canonical_execution_transaction FINAL
            WHERE meta_network_name = {network:String}
              AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
            GROUP BY block_number
        )
        SELECT
            toDate(b.block_date_time) AS date,
            sum(ifNull(t.eip7976_floor_uplift_current_body, 0))
                AS eip7976_floor_uplift_current_body,
            sum(ifNull(t.eip7976_floor_bound_txs_current_body, 0))
                AS eip7976_floor_bound_txs_current_body
        FROM default.canonical_execution_block AS b FINAL
        GLOBAL LEFT JOIN tx_by_block AS t USING block_number
        WHERE b.meta_network_name = {network:String}
          AND b.block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
        GROUP BY date
        ORDER BY date
        """,
        parameters={
            "network": NETWORK,
            "min_block": bounds["min_block"],
            "max_block": bounds["max_block"],
        },
        settings={"max_execution_time": 600},
    )


def query_state_core_daily(raw_client, bounds):
    zero_word = "0x" + "00" * 32
    return raw_client.query_df(
        """
        WITH
            blocks AS
            (
                SELECT block_number, toDate(block_date_time) AS date
                FROM default.canonical_execution_block FINAL
                WHERE meta_network_name = {network:String}
                  AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
            ),
            storage_by_block AS
            (
                SELECT
                    block_number,
                    uniqExact(tuple(lower(address), lower(slot)))
                        AS new_storage_slots
                FROM default.canonical_execution_storage_diffs FINAL
                WHERE meta_network_name = {network:String}
                  AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
                  AND lower(from_value) = {zero_word:String}
                  AND lower(to_value) != {zero_word:String}
                GROUP BY block_number
            ),
            code_by_block AS
            (
                SELECT
                    block_number,
                    sum(n_code_bytes) AS code_bytes
                FROM default.canonical_execution_contracts FINAL
                WHERE meta_network_name = {network:String}
                  AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
                GROUP BY block_number
            )
        SELECT
            b.date,
            sum(ifNull(s.new_storage_slots, 0)) AS new_storage_slots,
            sum(ifNull(c.code_bytes, 0)) AS code_bytes
        FROM blocks AS b
        GLOBAL LEFT JOIN storage_by_block AS s USING block_number
        GLOBAL LEFT JOIN code_by_block AS c USING block_number
        GROUP BY date
        ORDER BY date
        """,
        parameters={
            "network": NETWORK,
            "min_block": bounds["min_block"],
            "max_block": bounds["max_block"],
            "zero_word": zero_word,
        },
        settings={"max_execution_time": 600},
    )


def refresh_rpc_calibration(anchor_base):
    rpc_url, rpc_headers, provider = rpc_configuration()
    plan_path = DATA_DIR / (
        f"calibration_sample_plan_{ANCHOR_START}_{END_DATE}"
        f"_n{RPC_PLAN_PER_DAY}_seed{RPC_SEED}.csv"
    )
    rpc_path = DATA_DIR / (
        f"calibration_rpc_state_access_auth_blocks_{ANCHOR_START}_{END_DATE}.csv"
    )

    plan = sample_blocks_per_day(
        anchor_base[["date", "min_block", "max_block"]],
        n_per_day=RPC_PLAN_PER_DAY,
        seed=RPC_SEED,
    )
    plan.to_csv(plan_path, index=False)
    targets = plan[plan["sample_rank"] < RPC_BLOCKS_PER_DAY].copy()

    if rpc_path.exists():
        completed = pd.read_csv(rpc_path)
    else:
        completed = pd.DataFrame(columns=["block_number"])
    done = set(completed["block_number"].astype("int64")) if len(completed) else set()
    todo = [
        int(block)
        for block in targets["block_number"]
        if int(block) not in done
    ]

    print(
        f"RPC provider: {provider}; targets: {len(targets):,}; "
        f"cached: {len(done):,}; todo: {len(todo):,}"
    )
    failures = []
    for start in range(0, len(todo), RPC_CHUNK_SIZE):
        chunk = todo[start : start + RPC_CHUNK_SIZE]
        frame, failed = calibrate_blocks(
            rpc_url,
            chunk,
            cpsb=CPSB,
            include_reads=False,
            include_system_changes=False,
            include_bal=False,
            rpc_headers=rpc_headers,
            max_workers=RPC_MAX_WORKERS,
            on_error="skip",
        )
        failures.extend(failed)
        if not frame.empty:
            completed = pd.concat([completed, frame], ignore_index=True)
            completed = completed.drop_duplicates("block_number")
            completed.to_csv(rpc_path, index=False)
        print(
            f"RPC chunk {start}:{start + len(chunk)}; "
            f"cached rows {len(completed):,}; failures {len(failed)}"
        )
    if failures:
        raise RuntimeError(
            "RPC refresh finished with failed blocks; rerun explicitly after "
            f"reviewing the endpoint. First failures: {failures[:10]}"
        )
    return targets, completed


def build_calibrated_anchor(daily_full, raw_client):
    anchor_daily = daily_full[
        (daily_full["date"] >= pd.Timestamp(ANCHOR_START))
        & (daily_full["date"] < pd.Timestamp(END_DATE))
    ].copy()
    bounds = {
        "min_block": int(anchor_daily["min_block"].min()),
        "max_block": int(anchor_daily["max_block"].max()),
    }

    floor = normalize_dates(query_eip7976_daily(raw_client, bounds))
    state_core = normalize_dates(query_state_core_daily(raw_client, bounds))
    anchor_base = (
        anchor_daily.merge(floor, on="date", how="left", validate="one_to_one")
        .merge(state_core, on="date", how="left", validate="one_to_one")
    )
    anchor_base["current_gas_used"] = anchor_base["block_gas_used"]

    targets, rpc = refresh_rpc_calibration(anchor_base)
    rpc["block_number"] = rpc["block_number"].astype("int64")
    calibration = targets.merge(rpc, on="block_number", how="inner")

    daily_rates = calibration.groupby("date", as_index=False).agg(
        sampled_blocks=("block_number", "count"),
        tx_access_list_gas_7981_per_block=(
            "tx_access_list_gas_7981",
            "mean",
        ),
        new_accounts_per_block=("rpc_new_accounts", "mean"),
        new_delegation_indicators_per_block=(
            "rpc_new_delegation_indicators",
            "mean",
        ),
    )
    panel = anchor_base.merge(
        daily_rates,
        on="date",
        how="left",
        validate="one_to_one",
    )

    pooled = {
        "tx_access_list_gas_7981_per_block":
            calibration["tx_access_list_gas_7981"].mean(),
        "new_accounts_per_block": calibration["rpc_new_accounts"].mean(),
        "new_delegation_indicators_per_block":
            calibration["rpc_new_delegation_indicators"].mean(),
    }
    for column, fallback in pooled.items():
        panel[column] = panel[column].fillna(fallback)

    panel["cal_new_accounts"] = (
        panel["new_accounts_per_block"] * panel["block_count"]
    )
    panel["cal_new_delegation_indicators"] = (
        panel["new_delegation_indicators_per_block"] * panel["block_count"]
    )
    panel["current_state_creation_gas_calibrated"] = (
        20_000 * panel["new_storage_slots"]
        + 25_000 * panel["cal_new_accounts"]
        + 200 * panel["code_bytes"]
        + 12_500 * panel["cal_new_delegation_indicators"]
    )
    panel["state_gas_8037_calibrated"] = CPSB * (
        64 * panel["new_storage_slots"]
        + 120 * panel["cal_new_accounts"]
        + panel["code_bytes"]
        + 23 * panel["cal_new_delegation_indicators"]
    )
    panel["eip7981_access_list_data_gas_calibrated"] = (
        panel["tx_access_list_gas_7981_per_block"] * panel["block_count"]
    )

    out_path = DATA_DIR / (
        f"daily_accounting_panel_calibrated_no_bal_{ANCHOR_START}_{END_DATE}.csv"
    )
    panel.to_csv(out_path, index=False)
    raw_path = RAW_CACHE_DIR / (
        f"anchor_xatu_base_{ANCHOR_START}_{END_DATE}.csv"
    )
    anchor_base.to_csv(raw_path, index=False)
    return panel


if REFRESH_FROM_NETWORK:
    refreshed_anchor = build_calibrated_anchor(refreshed_daily, raw_client)
    print("refreshed calibrated anchor rows:", len(refreshed_anchor))
else:
    refreshed_anchor = None
    print("RPC refresh skipped; the deterministic cached sample will be used.")

RPC refresh skipped; the deterministic cached sample will be used.


## Xatu execution traces and refund reconstruction

Execution repricing requires full-range daily aggregates from three Xatu datasets:

- canonical structlog aggregates for SSTORE, cold-access, opcode, and refund counters;
- canonical traces for internal value transfers and contract creation; and
- raw transactions for EIP-2780 transaction paths.

RPC supplies access-list and authorization counts for the same deterministic sample. Refund-positive transactions are reconstructed from Xatu in two-day chunks using `sim.xatu_refunds.query_xatu_refund_daily_full`. Each completed day is cached immediately.

In [5]:
EXECUTION_TAG = f"{ANCHOR_START}_{END_DATE}"
OPS_CACHE = DATA_DIR / f"xatu_daily_execution_repricing_ops_{EXECUTION_TAG}.csv"
CALLS_CACHE = DATA_DIR / f"xatu_daily_execution_repricing_calls_{EXECUTION_TAG}.csv"
TX_TYPES_CACHE = DATA_DIR / f"xatu_daily_eip2780_transaction_types_{EXECUTION_TAG}.csv"
REFUND_CACHE = DATA_DIR / f"xatu_full_refund_daily_{EXECUTION_TAG}.csv"
EXECUTION_OUT = DATA_DIR / f"execution_repricing_daily_{EXECUTION_TAG}.csv"

OPS_QUERY = """
WITH op_by_block AS
(
    SELECT
        block_number,
        uniqExact(transaction_hash) AS traced_txs,
        sumIf(opcode_count, operation = 'SSTORE') AS sstore_count,
        sumIf(gas, operation = 'SSTORE') AS sstore_gas_current,
        sumIf(cold_access_count, operation = 'SSTORE') AS sstore_cold_count,
        sumIf(cold_access_count, operation = 'SLOAD') AS sload_cold_count,
        sumIf(cold_access_count, operation IN (
            'BALANCE', 'EXTCODEHASH', 'EXTCODESIZE', 'EXTCODECOPY',
            'CALL', 'CALLCODE', 'DELEGATECALL', 'STATICCALL', 'SELFDESTRUCT'
        )) AS account_cold_count,
        sumIf(opcode_count, operation IN ('EXTCODESIZE', 'EXTCODECOPY'))
            AS ext_second_read_count,
        sumIf(opcode_count, operation IN ('CREATE', 'CREATE2'))
            AS internal_create_opcode_count,
        sumIf(opcode_count, operation = 'SELFDESTRUCT') AS selfdestruct_count,
        sumIf(ifNull(gas_refund, 0), operation = '' AND call_frame_id = 0)
            AS refund_counter_current
    FROM default.canonical_execution_transaction_structlog_agg FINAL
    WHERE meta_network_name = {network:String}
      AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
    GROUP BY block_number
)
SELECT
    toDate(b.block_date_time) AS date,
    countIf(o.traced_txs > 0) AS blocks_with_traces,
    sum(ifNull(o.traced_txs, 0)) AS traced_txs,
    sum(ifNull(o.sstore_count, 0)) AS sstore_count,
    sum(ifNull(o.sstore_gas_current, 0)) AS sstore_gas_current,
    sum(ifNull(o.sstore_cold_count, 0)) AS sstore_cold_count,
    sum(ifNull(o.sload_cold_count, 0)) AS sload_cold_count,
    sum(ifNull(o.account_cold_count, 0)) AS account_cold_count,
    sum(ifNull(o.ext_second_read_count, 0)) AS ext_second_read_count,
    sum(ifNull(o.internal_create_opcode_count, 0))
        AS internal_create_opcode_count,
    sum(ifNull(o.selfdestruct_count, 0)) AS selfdestruct_count,
    sum(ifNull(o.refund_counter_current, 0)) AS refund_counter_current
FROM default.canonical_execution_block AS b FINAL
GLOBAL LEFT JOIN op_by_block AS o USING block_number
WHERE b.meta_network_name = {network:String}
  AND b.block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
GROUP BY date
ORDER BY date
"""

CALLS_QUERY = """
WITH calls_by_block AS
(
    SELECT
        block_number,
        countIf(
            action_type = 'call'
            AND trace_address IS NOT NULL
            AND action_call_type IN ('call', 'call_code')
            AND action_value > 0
        ) AS positive_value_internal_calls,
        countIf(
            action_type = 'create' AND trace_address IS NOT NULL
        ) AS internal_create_traces,
        countIf(
            action_type = 'create'
            AND trace_address IS NOT NULL
            AND error IS NULL
            AND result_address IS NOT NULL
        ) AS successful_internal_creates
    FROM default.canonical_execution_traces FINAL
    WHERE meta_network_name = {network:String}
      AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
    GROUP BY block_number
)
SELECT
    toDate(b.block_date_time) AS date,
    sum(ifNull(c.positive_value_internal_calls, 0))
        AS positive_value_internal_calls,
    sum(ifNull(c.internal_create_traces, 0)) AS internal_create_traces,
    sum(ifNull(c.successful_internal_creates, 0))
        AS successful_internal_creates
FROM default.canonical_execution_block AS b FINAL
GLOBAL LEFT JOIN calls_by_block AS c USING block_number
WHERE b.meta_network_name = {network:String}
  AND b.block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
GROUP BY date
ORDER BY date
"""

TX_TYPES_QUERY = """
WITH tx_by_block AS
(
    SELECT
        block_number,
        count() AS tx_count_trace_input,
        countIf(`to` IS NOT NULL AND lower(`from`) = lower(`to`))
            AS self_txs,
        countIf(
            `to` IS NOT NULL
            AND lower(`from`) != lower(`to`)
            AND value = 0
        ) AS distinct_zero_value_txs,
        countIf(
            `to` IS NOT NULL
            AND lower(`from`) != lower(`to`)
            AND value > 0
        ) AS distinct_value_txs,
        countIf(`to` IS NULL AND success AND value = 0)
            AS create_success_zero_value,
        countIf(`to` IS NULL AND success AND value > 0)
            AS create_success_value,
        countIf(`to` IS NULL AND NOT success AND value = 0)
            AS create_failed_zero_value,
        countIf(`to` IS NULL AND NOT success AND value > 0)
            AS create_failed_value,
        countIf(type = 4) AS type4_txs
    FROM default.execution_transaction FINAL
    WHERE meta_network_name = {network:String}
      AND block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
    GROUP BY block_number
)
SELECT
    toDate(b.block_date_time) AS date,
    sum(ifNull(t.tx_count_trace_input, 0)) AS tx_count_trace_input,
    sum(ifNull(t.self_txs, 0)) AS self_txs,
    sum(ifNull(t.distinct_zero_value_txs, 0)) AS distinct_zero_value_txs,
    sum(ifNull(t.distinct_value_txs, 0)) AS distinct_value_txs,
    sum(ifNull(t.create_success_zero_value, 0)) AS create_success_zero_value,
    sum(ifNull(t.create_success_value, 0)) AS create_success_value,
    sum(ifNull(t.create_failed_zero_value, 0)) AS create_failed_zero_value,
    sum(ifNull(t.create_failed_value, 0)) AS create_failed_value,
    sum(ifNull(t.type4_txs, 0)) AS type4_txs
FROM default.canonical_execution_block AS b FINAL
GLOBAL LEFT JOIN tx_by_block AS t USING block_number
WHERE b.meta_network_name = {network:String}
  AND b.block_number BETWEEN {min_block:UInt64} AND {max_block:UInt64}
GROUP BY date
ORDER BY date
"""


def query_execution_source(raw_client, query, bounds):
    return raw_client.query_df(
        query,
        parameters={
            "network": NETWORK,
            "min_block": int(bounds["min_block"]),
            "max_block": int(bounds["max_block"]),
        },
        settings={"max_execution_time": 900},
    )


def refresh_execution_aggregates(raw_client, bounds):
    outputs = {}
    for name, path, query in [
        ("ops", OPS_CACHE, OPS_QUERY),
        ("calls", CALLS_CACHE, CALLS_QUERY),
        ("tx_types", TX_TYPES_CACHE, TX_TYPES_QUERY),
    ]:
        frame = normalize_dates(
            query_execution_source(raw_client, query, bounds)
        )
        frame.to_csv(path, index=False)
        outputs[name] = frame
        print("wrote", path.relative_to(PROJECT_ROOT))
    return outputs

In [6]:
def build_execution_repricing(anchor_panel, raw_client):
    rpc_path = DATA_DIR / (
        f"calibration_rpc_state_access_auth_blocks_{ANCHOR_START}_{END_DATE}.csv"
    )
    rpc = pd.read_csv(rpc_path).sort_values("block_number")
    rpc["block_number"] = rpc["block_number"].astype("int64")

    accounting = anchor_panel.copy()
    accounting["date"] = pd.to_datetime(accounting["date"])
    accounting["execution_current"] = (
        accounting["current_gas_used"]
        - accounting["data_gas_current"]
        - accounting["current_state_creation_gas_calibrated"]
    )
    assert accounting["execution_current"].gt(0).all()

    day_bounds = accounting[
        ["date", "min_block", "max_block"]
    ].sort_values("min_block")
    rpc = pd.merge_asof(
        rpc,
        day_bounds,
        left_on="block_number",
        right_on="min_block",
        direction="backward",
    )
    if rpc["date"].isna().any() or (
        rpc["block_number"] > rpc["max_block"]
    ).any():
        raise ValueError("RPC sample block could not be mapped to an anchor day")

    rpc["eoa_account_write_count"] = (
        rpc["authorization_set_tuple_count"]
        + rpc["authorization_clear_tuple_count"]
    )
    sample_daily = rpc.groupby("date", as_index=False).agg(
        rpc_sampled_blocks=("block_number", "nunique"),
        access_list_address_count_sample=(
            "tx_access_list_address_count",
            "sum",
        ),
        access_list_key_count_sample=(
            "tx_access_list_storage_key_count",
            "sum",
        ),
        eoa_account_write_count_sample=("eoa_account_write_count", "sum"),
    )
    for name in [
        "access_list_address_count",
        "access_list_key_count",
        "eoa_account_write_count",
    ]:
        sample_daily[f"{name}_per_block"] = (
            sample_daily[f"{name}_sample"]
            / sample_daily["rpc_sampled_blocks"]
        )

    accounting = accounting.merge(
        sample_daily[
            [
                "date",
                "rpc_sampled_blocks",
                "access_list_address_count_per_block",
                "access_list_key_count_per_block",
                "eoa_account_write_count_per_block",
            ]
        ],
        on="date",
        how="left",
        validate="one_to_one",
    )
    rpc_source_columns = {
        "access_list_address_count": "tx_access_list_address_count",
        "access_list_key_count": "tx_access_list_storage_key_count",
        "eoa_account_write_count": "eoa_account_write_count",
    }
    for name, source in rpc_source_columns.items():
        fallback = rpc[source].sum() / rpc["block_number"].nunique()
        accounting[f"{name}_per_block"] = (
            accounting[f"{name}_per_block"].fillna(fallback)
        )
        accounting[name] = (
            accounting[f"{name}_per_block"] * accounting["block_count"]
        )

    bounds = {
        "min_block": int(accounting["min_block"].min()),
        "max_block": int(accounting["max_block"].max()),
    }
    extracts = refresh_execution_aggregates(raw_client, bounds)
    daily = (
        accounting.merge(
            extracts["ops"],
            on="date",
            how="left",
            validate="one_to_one",
        )
        .merge(
            extracts["calls"],
            on="date",
            how="left",
            validate="one_to_one",
        )
        .merge(
            extracts["tx_types"],
            on="date",
            how="left",
            validate="one_to_one",
        )
    )

    trace_columns = (
        list(extracts["ops"].columns.drop("date"))
        + list(extracts["calls"].columns.drop("date"))
        + list(extracts["tx_types"].columns.drop("date"))
    )
    if daily[trace_columns].isna().any().any():
        raise ValueError("Xatu execution extracts do not cover every anchor day")
    daily[trace_columns] = daily[trace_columns].astype("float64")

    WARM_ACCESS = 100
    CURRENT_COLD_STORAGE = 2_100
    CURRENT_FIRST_CHANGE = 2_800
    CURRENT_NEW_SLOT_EXTRA = 17_100
    CURRENT_STORAGE_CREATION = 20_000
    CURRENT_NEW_ACCOUNT = 25_000
    CURRENT_CREATE = 32_000

    COLD_ACCOUNT_8038 = 3_000
    COLD_STORAGE_8038 = 3_000
    STORAGE_WRITE_8038 = 10_000
    CREATE_ACCESS_8038 = 11_000

    daily["sstore_first_changes_unclipped"] = (
        daily["sstore_gas_current"]
        - WARM_ACCESS * daily["sstore_count"]
        - CURRENT_COLD_STORAGE * daily["sstore_cold_count"]
        - CURRENT_NEW_SLOT_EXTRA * daily["new_storage_slots"]
    ) / CURRENT_FIRST_CHANGE
    daily["sstore_first_changes_est"] = (
        daily["sstore_first_changes_unclipped"].clip(
            lower=daily["new_storage_slots"],
            upper=daily["sstore_count"],
        )
    )
    daily["sstore_reconstructed_current"] = (
        WARM_ACCESS * daily["sstore_count"]
        + CURRENT_COLD_STORAGE * daily["sstore_cold_count"]
        + CURRENT_FIRST_CHANGE * daily["sstore_first_changes_est"]
        + CURRENT_NEW_SLOT_EXTRA * daily["new_storage_slots"]
    )
    daily["sstore_reconstruction_residual"] = (
        daily["sstore_gas_current"]
        - daily["sstore_reconstructed_current"]
    )
    daily["sstore_regular_current_aligned"] = (
        daily["sstore_gas_current"]
        - CURRENT_STORAGE_CREATION * daily["new_storage_slots"]
    )
    daily["sstore_regular_8038"] = (
        WARM_ACCESS * (daily["sstore_count"] - daily["sstore_cold_count"])
        + COLD_STORAGE_8038 * daily["sstore_cold_count"]
        + STORAGE_WRITE_8038 * daily["sstore_first_changes_est"]
    )

    daily["delta_8038_sstore"] = (
        daily["sstore_regular_8038"]
        - daily["sstore_regular_current_aligned"]
    )
    daily["delta_8038_sload_cold"] = 900 * daily["sload_cold_count"]
    daily["delta_8038_account_cold"] = 400 * daily["account_cold_count"]
    daily["delta_8038_ext_second_read"] = (
        100 * daily["ext_second_read_count"]
    )
    daily["delta_8038_internal_value_calls"] = (
        1_300 * daily["positive_value_internal_calls"]
    )
    daily["delta_8038_access_lists"] = (
        600 * daily["access_list_address_count"]
        + 1_100 * daily["access_list_key_count"]
    )
    daily["delta_8038_authorizations"] = (
        1_300 * daily["eoa_account_write_count"]
    )
    daily["delta_8038_internal_create"] = (
        (CREATE_ACCESS_8038 - CURRENT_CREATE)
        * daily["internal_create_opcode_count"]
        + CURRENT_NEW_ACCOUNT * daily["successful_internal_creates"]
    )
    daily["delta_8038_selfdestruct_unmeasured_max"] = (
        1_300 * daily["selfdestruct_count"]
    )

    delta_columns = [
        "delta_8038_sstore",
        "delta_8038_sload_cold",
        "delta_8038_account_cold",
        "delta_8038_ext_second_read",
        "delta_8038_internal_value_calls",
        "delta_8038_access_lists",
        "delta_8038_authorizations",
        "delta_8038_internal_create",
    ]
    daily["delta_8038_central"] = daily[delta_columns].sum(axis=1)
    daily["delta_2780_intrinsic"] = (
        -9_000 * daily["self_txs"]
        - 6_000 * daily["distinct_zero_value_txs"]
        - 5_000 * daily["create_success_zero_value"]
        - 3_244 * daily["create_success_value"]
        - 30_000 * daily["create_failed_zero_value"]
        - 28_244 * daily["create_failed_value"]
    )
    daily["delta_requested_central"] = (
        daily["delta_8038_central"] + daily["delta_2780_intrinsic"]
    )
    daily["execution_8038_2780"] = (
        daily["execution_current"] + daily["delta_requested_central"]
    )

    storage_state_new = (
        daily["new_storage_slots"] * 64 * CPSB
    )
    storage_state_current = (
        daily["new_storage_slots"] * CURRENT_STORAGE_CREATION
    )
    other_regular_delta = (
        daily["delta_requested_central"] - daily["delta_8038_sstore"]
    )
    other_state_delta = (
        daily["state_gas_8037_calibrated"]
        - daily["current_state_creation_gas_calibrated"]
        - (storage_state_new - storage_state_current)
    )
    other_gas_base = (
        daily["current_gas_used"] - daily["sstore_gas_current"]
    ).clip(lower=1)
    other_gross_delta_rate = (
        other_regular_delta.sum() + other_state_delta.sum()
    ) / other_gas_base.sum()

    completed = pd.DataFrame()
    if REFUND_CACHE.exists():
        completed = normalize_dates(pd.read_csv(REFUND_CACHE))
    completed_dates = (
        set(completed["date"]) if not completed.empty else set()
    )
    pending = daily[
        ~daily["date"].isin(completed_dates)
    ].sort_values("date")

    for start in range(0, len(pending), 2):
        chunk = pending.iloc[start : start + 2]
        result = query_xatu_refund_daily_full(
            raw_client,
            min_block=int(chunk["min_block"].min()),
            max_block=int(chunk["max_block"].max()),
            other_gross_delta_rate=float(other_gross_delta_rate),
            network=NETWORK,
        )
        result["date"] = pd.to_datetime(result["date"])
        completed = pd.concat([completed, result], ignore_index=True)
        completed = (
            completed.drop_duplicates("date", keep="last")
            .sort_values("date")
        )
        completed.to_csv(REFUND_CACHE, index=False)
        print(
            "refund days cached:",
            len(completed),
            "/",
            len(daily),
        )

    refund_daily = normalize_dates(pd.read_csv(REFUND_CACHE))
    refund_columns = [
        column for column in refund_daily.columns if column != "date"
    ]
    refund_daily[refund_columns] = refund_daily[refund_columns].apply(
        pd.to_numeric,
        errors="raise",
    )
    daily = daily.merge(
        refund_daily,
        on="date",
        how="left",
        validate="one_to_one",
    )
    if daily[refund_columns].isna().any().any():
        raise ValueError("Refund cache does not cover every anchor day")

    daily["refund_proxy_scale"] = (
        daily["refund_counter_total"]
        / daily["refund_counter_identified"]
    )
    daily["refund_extra_8038"] = (
        daily["extra_refund_current_floor"]
        * daily["refund_proxy_scale"]
    )
    daily["execution_8038_2780_refund_corrected"] = (
        daily["execution_8038_2780"]
        - daily["refund_extra_8038"]
    )
    daily["m_execution_8038_2780_refund_corrected_daily"] = (
        daily["execution_8038_2780_refund_corrected"]
        / daily["execution_current"]
    )

    assert np.allclose(
        daily["refund_counter_total"],
        daily["refund_counter_current"],
        rtol=0,
        atol=0,
    )
    daily.to_csv(EXECUTION_OUT, index=False)
    print("wrote", EXECUTION_OUT.relative_to(PROJECT_ROOT))
    return daily


if REFRESH_FROM_NETWORK:
    refreshed_execution = build_execution_repricing(
        refreshed_anchor,
        raw_client,
    )
    print("refreshed execution-repricing rows:", len(refreshed_execution))
else:
    refreshed_execution = None
    print("Execution Xatu/refund refresh skipped; cached export will be used.")

Execution Xatu/refund refresh skipped; cached export will be used.


## Exported inputs

The 2025 event panel already contains daily block, fee, calldata, and state-inventory measurements. The January–May extension is reconstructed from exported daily Xatu aggregates. The 120-day anchor uses the no-BAL calibrated accounting panel because BAL data are not part of the Glamsterdam mechanism analyzed in the report.

In [7]:
SOURCE_FILES = {
    "event_panel": DATA_DIR / "gas_limit_data_share_daily_2025-01-01_2026-01-05.csv",
    "january_data": DATA_DIR / "daily_current_data_gas_xatu_2026-01-05_2026-02-01.csv",
    "anchor_data": DATA_DIR / "daily_current_data_gas_xatu_2026-02-01_2026-06-01.csv",
    "january_accounting": DATA_DIR / "daily_accounting_panel_2026-01-05_2026-02-01.csv",
    "anchor_accounting_no_bal": DATA_DIR / "daily_accounting_panel_calibrated_no_bal_2026-02-01_2026-06-01.csv",
    "execution_repricing": DATA_DIR / "execution_repricing_daily_2026-02-01_2026-06-01.csv",
}

missing = [str(path) for path in SOURCE_FILES.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing cached inputs. Restore them under data/ or set "
        "REFRESH_FROM_NETWORK = True above to rebuild them:\n- "
        + "\n- ".join(missing)
    )

manifest = pd.DataFrame([
    {
        "artifact": name,
        "path_relative_to_repo": str(path.relative_to(PROJECT_ROOT)),
        "bytes": path.stat().st_size,
        "role": {
            "event_panel": "2025 gas-limit event resource panel",
            "january_data": "January 2026 current calldata/data-gas export",
            "anchor_data": "February-May 2026 current calldata/data-gas export",
            "january_accounting": "January 2026 state-inventory and fee bridge",
            "anchor_accounting_no_bal": "120-day calibrated state/access-list accounting",
            "execution_repricing": "120-day EIP-8038/EIP-2780 execution replay",
        }[name],
    }
    for name, path in SOURCE_FILES.items()
])
manifest.to_csv(MANIFEST_OUT, index=False)
manifest

,artifact,path_relative_to_repo,bytes,role
0,event_panel,data/gas_limit_data_share_daily_2025-01-01_202...,235574,2025 gas-limit event resource panel
1,january_data,data/daily_current_data_gas_xatu_2026-01-05_20...,3408,January 2026 current calldata/data-gas export
2,anchor_data,data/daily_current_data_gas_xatu_2026-02-01_20...,14466,February-May 2026 current calldata/data-gas ex...
3,january_accounting,data/daily_accounting_panel_2026-01-05_2026-02...,9132,January 2026 state-inventory and fee bridge
4,anchor_accounting_no_bal,data/daily_accounting_panel_calibrated_no_bal_...,136224,120-day calibrated state/access-list accounting
5,execution_repricing,data/execution_repricing_daily_2026-02-01_2026...,283144,120-day EIP-8038/EIP-2780 execution replay


## Daily resource panel

The mutually exclusive historical quantities are measured in historical gas-equivalent units:

- data is current-rule calldata gas, including the EIP-7623 floor;
- state is the calibrated inventory-delta proxy used by the event study; and
- execution is total receipt gas minus data and state, retaining the historical intrinsic transaction charge.

The January–May extension uses the same accounting convention as the 2025 panel.

In [8]:
event = pd.read_csv(SOURCE_FILES["event_panel"])
event["date"] = pd.to_datetime(event["date"])
event["q_data"] = event["data_gas_current"].astype(float)
event["q_state"] = event["current_state_inventory_delta_gas_proxy"].astype(float)
event["q_execution"] = (
    event["block_gas_used"].astype(float) - event["q_data"] - event["q_state"]
).clip(lower=0)
event["segment"] = "event_panel"

extension_data = pd.concat(
    [
        pd.read_csv(SOURCE_FILES["january_data"]),
        pd.read_csv(SOURCE_FILES["anchor_data"]),
    ],
    ignore_index=True,
)
extension_accounting = pd.concat(
    [
        pd.read_csv(SOURCE_FILES["january_accounting"]),
        pd.read_csv(SOURCE_FILES["anchor_accounting_no_bal"]),
    ],
    ignore_index=True,
)
for frame in (extension_data, extension_accounting):
    frame["date"] = pd.to_datetime(frame["date"])

extension = extension_data.merge(
    extension_accounting[
        [
            "date",
            "median_base_fee_per_gas",
            "current_gas_used",
            "mean_gas_limit",
            "accounts_net_delta",
            "storage_bytes_net_delta",
            "contract_code_bytes_net_delta",
        ]
    ],
    on="date",
    how="inner",
    validate="one_to_one",
)
extension["q_data"] = extension["data_gas_current"].astype(float)
extension["q_state"] = (
    extension["accounts_net_delta"].clip(lower=0) * 25_000
    + extension["storage_bytes_net_delta"].clip(lower=0) * (20_000 / 32)
    + extension["contract_code_bytes_net_delta"].clip(lower=0) * 200
)
extension["q_execution"] = (
    extension["receipt_gas_used"] - extension["q_data"] - extension["q_state"]
).clip(lower=0)
extension["block_gas_used"] = extension["receipt_gas_used"]
extension["gas_limit_mgas"] = (extension["mean_gas_limit"] / 1e6).round()
extension["segment"] = "2026_extension"

keep = [
    "date",
    "segment",
    "block_count",
    "block_gas_used",
    "median_base_fee_per_gas",
    "gas_limit_mgas",
    "q_execution",
    "q_data",
    "q_state",
]
daily = pd.concat([event[keep], extension[keep]], ignore_index=True)
daily = daily.sort_values("date").reset_index(drop=True)
daily["q_total"] = daily[["q_execution", "q_data", "q_state"]].sum(axis=1)
for resource in ["execution", "data", "state"]:
    daily[f"s_{resource}"] = daily[f"q_{resource}"] / daily["q_total"]
daily["log_odds_data"] = np.log(daily["q_data"] / daily["q_execution"])
daily["log_odds_state"] = np.log(daily["q_state"] / daily["q_execution"])
daily["log_fee"] = np.log(daily["median_base_fee_per_gas"])
daily["gas_per_block"] = daily["block_gas_used"] / daily["block_count"]

expected_dates = pd.date_range("2025-01-01", "2026-05-31", freq="D")
assert list(daily["date"]) == list(expected_dates)
assert daily["date"].is_unique
assert (daily[["q_execution", "q_data", "q_state"]] > 0).all().all()
assert np.allclose(daily["q_total"], daily["block_gas_used"], rtol=1e-10, atol=1e-3)

daily.to_csv(DAILY_OUT, index=False)
print(f"Wrote {DAILY_OUT.relative_to(PROJECT_ROOT)} with {len(daily)} daily rows")

Wrote data/glamsterdam/daily_resource_panel_2025-01-01_2026-05-31.csv with 516 daily rows


In [9]:
total_blocks = daily["block_count"].sum()
resource_mix = pd.DataFrame([
    {
        "resource": resource,
        "mean_gas_per_block": daily[f"q_{resource}"].sum() / total_blocks,
        "share": daily[f"q_{resource}"].sum() / daily["q_total"].sum(),
    }
    for resource in ["execution", "data", "state"]
])
resource_mix

,resource,mean_gas_per_block,share
0,execution,"17,329,481.411875",0.735724
1,data,"809,114.647085",0.034351
2,state,"5,415,720.077168",0.229925


## February–May 2026 anchor panel

This join keeps only the fields used by the metering and equilibrium notebooks. In particular, it uses the calibrated no-BAL accounting panel and joins the exported current-data and execution-repricing results by date.

In [10]:
anchor = pd.read_csv(SOURCE_FILES["anchor_accounting_no_bal"])
current_data = pd.read_csv(SOURCE_FILES["anchor_data"])
execution = pd.read_csv(SOURCE_FILES["execution_repricing"])

for frame in (anchor, current_data, execution):
    frame["date"] = pd.to_datetime(frame["date"])

anchor_columns = [
    "date",
    "block_count",
    "current_gas_used",
    "median_base_fee_per_gas",
    "eip7976_floor_uplift_current_body",
    "current_state_creation_gas_calibrated",
    "state_gas_8037_calibrated",
    "eip7981_access_list_data_gas_calibrated",
]
current_columns = [
    "date",
    "data_gas_current",
    "eip7623_data_uplift_proxy",
]
execution_columns = [
    "date",
    "execution_current",
    "execution_8038_2780",
    "execution_8038_2780_refund_corrected",
    "m_execution_8038_2780_refund_corrected_daily",
    "delta_8038_sstore",
    "delta_8038_sload_cold",
    "delta_8038_account_cold",
    "delta_8038_ext_second_read",
    "delta_8038_internal_value_calls",
    "delta_8038_access_lists",
    "delta_8038_authorizations",
    "delta_8038_internal_create",
    "delta_8038_central",
    "delta_2780_intrinsic",
    "refund_extra_8038",
    "refund_txs_total",
    "refund_counter_total",
    "refund_txs_identified",
    "refund_counter_identified",
]

anchor = (
    anchor[anchor_columns]
    .merge(current_data[current_columns], on="date", how="inner", validate="one_to_one")
    .merge(execution[execution_columns], on="date", how="inner", validate="one_to_one")
    .sort_values("date")
    .reset_index(drop=True)
)
anchor["data_gas_glamsterdam"] = (
    anchor["data_gas_current"]
    + anchor["eip7976_floor_uplift_current_body"]
    + anchor["eip7981_access_list_data_gas_calibrated"]
)
anchor["historical_execution_gas"] = (
    anchor["current_gas_used"]
    - anchor["data_gas_current"]
    - anchor["current_state_creation_gas_calibrated"]
)

assert len(anchor) == 120
assert list(anchor["date"]) == list(pd.date_range("2026-02-01", "2026-05-31", freq="D"))
required = [
    "data_gas_current",
    "data_gas_glamsterdam",
    "current_state_creation_gas_calibrated",
    "state_gas_8037_calibrated",
    "historical_execution_gas",
    "execution_8038_2780_refund_corrected",
]
assert not anchor[required].isna().any().any()
assert np.allclose(
    anchor["execution_current"].sum(),
    anchor["historical_execution_gas"].sum(),
    rtol=1e-10,
    atol=1,
)

anchor.to_csv(ANCHOR_OUT, index=False)
print(f"Wrote {ANCHOR_OUT.relative_to(PROJECT_ROOT)} with {len(anchor)} daily rows")
anchor.head(3)

Wrote data/glamsterdam/anchor_accounting_panel_2026-02-01_2026-05-31.csv with 120 daily rows


,date,block_count,current_gas_used,median_base_fee_per_gas,eip7976_floor_uplift_current_body,current_state_creation_gas_calibrated,state_gas_8037_calibrated,eip7981_access_list_data_gas_calibrated,data_gas_current,eip7623_data_uplift_proxy,...,delta_8038_internal_create,delta_8038_central,delta_2780_intrinsic,refund_extra_8038,refund_txs_total,refund_counter_total,refund_txs_identified,refund_counter_identified,data_gas_glamsterdam,historical_execution_gas
0,2026-02-01,7161,217479380384,"148,863,401.000000",4195293096,"33,448,728,250.000000","195,803,889,827.399994","3,881,578,229.760000",7030431652,768939216,...,"27,058,000.000000","102,132,207,180.857117","-7,361,487,656.000000","3,347,861,198.880614",552256,13665904100,551276,13656103000,"15,107,302,977.760000","177,000,220,482.000000"
1,2026-02-02,7160,217513933432,"182,441,382.000000",5205823870,"31,658,547,600.000000","187,172,845,596.000000","4,530,967,142.400001",7614086418,909574518,...,"38,825,000.000000","102,893,974,979.428558","-7,196,260,032.000000","3,592,829,265.386380",571173,13532393000,570116,13523042100,"17,350,877,430.400002","178,241,299,414.000000"
2,2026-02-03,7149,217550972098,"153,347,493.000000",5218017231,"34,348,498,450.000000","203,311,479,447.000000","3,924,048,353.280000",7778572268,1034409060,...,"41,265,000.000000","98,010,878,987.142838","-7,198,241,676.000000","3,331,078,347.838126",556249,12780689000,555040,12768238700,"16,920,637,852.280001","175,423,901,380.000000"


## Handoff

Notebook 2 reads the 120-day anchor panel to calculate the three Glamsterdam metering multipliers. Notebook 3 reads the 516-day resource panel to estimate the linked elasticity vectors. Both files are generated artifacts and remain under the ignored `data/` directory.